# Quickstart: Conditional Feature Importance with tcpfi

This notebook demonstrates how to use tcpfi for computing conditional feature importance in multi-time series forecasting.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

## 1. Create Sample Multi-Series Data

We'll create synthetic time series data for multiple stores with hourly observations.

In [ ]:
np.random.seed(42)
n_periods = 200
n_series = 3

# Create date range
dates = pd.date_range('2023-01-01', periods=n_periods, freq='h')

# Generate series data
series_data = {}
for i in range(n_series):
    series_id = f'store_{i+1:03d}'
    base = np.random.randn() * 10
    trend = np.linspace(0, 5, n_periods)
    seasonal = 5 * np.sin(2 * np.pi * np.arange(n_periods) / 24)
    noise = np.random.randn(n_periods) * 0.5
    series_data[series_id] = base + trend + seasonal + noise

df = pd.DataFrame(series_data, index=dates)
print(f"Data shape: {df.shape}")
df.head()

## 2. Create Lag Features Manually

For this example, we'll manually create lag features to demonstrate tcpfi without requiring skforecast.

In [ ]:
def create_lag_features(series: pd.Series, n_lags: int = 5) -> pd.DataFrame:
    """Create lag features from a time series."""
    df = pd.DataFrame()
    for lag in range(1, n_lags + 1):
        df[f'lag_{lag}'] = series.shift(lag)
    df['target'] = series
    return df.dropna()

# Create features for each series and combine
n_lags = 5
all_data = []

for series_id in df.columns:
    series_df = create_lag_features(df[series_id], n_lags)
    series_df['series_id'] = series_id
    series_df.index = pd.MultiIndex.from_arrays(
        [[series_id] * len(series_df), series_df.index],
        names=['level', 'date']
    )
    all_data.append(series_df)

combined = pd.concat(all_data)
X = combined[[f'lag_{i}' for i in range(1, n_lags + 1)]]
y = combined['target']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

## 3. Train a Global Model

In [ ]:
# Train a RandomForest model
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    random_state=42
)
model.fit(X.values, y.values)

# Check baseline performance
predictions = model.predict(X.values)
mse = np.mean((y.values - predictions) ** 2)
print(f"Training MSE: {mse:.4f}")

## 4. Compute Conditional Permutation Importance

Now let's use tcpfi to compute conditional permutation importance with automatic tree-based partitioning (cs-PFI).

In [ ]:
from tcpfi import ConditionalPermutationImportance

# Create explainer with automatic strategy (tree-based cs-PFI)
explainer = ConditionalPermutationImportance(
    model=model,
    metric='mse',
    strategy='auto',
    n_repeats=3,
    n_jobs=1,
    random_state=42,
)

# Compute importance
result = explainer.compute(X, y)

# Display results
result.to_dataframe()

## 5. Visualize Results

In [ ]:
from tcpfi.visualization import plot_importance_bar

fig, ax = plot_importance_bar(result, title='Conditional Permutation Feature Importance')

## 6. Using Manual Groups

You can also define custom groups based on domain knowledge.

In [ ]:
from tcpfi import ManualPartitioner

# Define groups based on domain knowledge
mapping = {
    'store_001': 'region_A',
    'store_002': 'region_B',
    'store_003': 'region_A',
}

partitioner = ManualPartitioner(mapping, series_col='level')

# Create explainer with manual partitioner
explainer_manual = ConditionalPermutationImportance(
    model=model,
    metric='mse',
    strategy='manual',
    partitioner=partitioner,
    n_repeats=3,
    n_jobs=1,
    random_state=42,
)

result_manual = explainer_manual.compute(X, y)
result_manual.to_dataframe()

## 7. Compare Methods

In [ ]:
from tcpfi.visualization import plot_importance_heatmap

results = {
    'Auto (cs-PFI)': result,
    'Manual (Region)': result_manual,
}

fig, ax = plot_importance_heatmap(results, title='Feature Importance Comparison')

## Summary

In this notebook, we demonstrated:

1. Creating multi-series time series data with lag features
2. Training a global RandomForest model
3. Computing conditional permutation importance using:
   - Automatic tree-based partitioning (cs-PFI)
   - Manual domain-based grouping
4. Visualizing and comparing results

The key insight is that `lag_1` (most recent lag) typically has the highest importance, which aligns with the autocorrelation structure of time series data.